## Estimador de RR desde PPG generalizable entre poblaciones (CASE, BIDMC, CapnoBase)
**Hallazgo previo (figura del espectro alineado):** la respiración es un pico **local** sobre un fondo de baja
frecuencia cuya forma depende de la población (plano en UCI/quirófano, descendente en sanos despiertos).
La FFT original elige el máximo **global** y por eso falla en CASE.

**Métodos** (módulo `rr_generalizable.py`, en la misma carpeta que este notebook):
- **A** FFT original · **D** conteo con pasa-alto fijo 0.17 Hz (ganador en CASE, notebook 12)
- **S** pico sobre el fondo (ajuste robusto log-log del fondo, pico de mayor prominencia sobre él)
- **Da** conteo con pasa-alto adaptativo = 0.6 × RR(S)
- **F** fusión de S y Da: media si coinciden ±2 rpm; si no, **abstención**

**Protocolo:** parámetros fijados de antemano e idénticos en las tres bases; mismas reglas de ventana válida
que en los notebooks 02, 03 y 09. Además: sensibilidad a los parámetros y validación dejando una base fuera.

In [ ]:
# g-1 — Imports, configuración y rutas
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wfdb
from scipy.signal import butter, decimate, filtfilt, find_peaks, resample_poly

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR / "notebooks"))
sys.path.insert(0, str(Path.cwd()))
import rr_generalizable as rg

OUT = BASE_DIR / "results"
CFG = json.loads((OUT / "config_congelada.json").read_text())
P = {**rg.PARAMS, "rpm_min": CFG["rpm_min"], "rpm_max": CFG["rpm_max"], "lp_riiv_hz": CFG["lp_fft_hz"],
     "lp_art_hz": CFG["lp_art_hz"], "umbral_art": CFG["umbral_art"]}     # mismos valores que la config congelada
FACTORES = (0.5, 0.6, 0.7)                 # sensibilidad del pasa-alto adaptativo
TOLS = (2.0, 3.0)                          # sensibilidad de la tolerancia de fusión
VENTANA_S, PASO_S, FS = rg.VENTANA_S, rg.PASO_S, rg.FS
CASE_DIR = BASE_DIR / "data" / "CASE" / "data" / "interpolated" / "physiological"
CAPNO_DIR = BASE_DIR / "data" / "capnobase" / "csv"
ACTIVACION = {1: 1, 2: 1, 7: 1, 8: 1, 3: 0, 4: 0, 5: 0, 6: 0}
pd.set_option("display.width", 250)
print(json.dumps({k: P[k] for k in ("rpm_min", "rpm_max", "hp_fijo_hz", "hp_factor", "tol_fusion_rpm")}))

In [ ]:
# g-2 — Referencias y ventanas de las tres bases (tarda ~10–15 min)
def rr_ref(m, t0, t1, min_marcas=3):
    m = m[(m >= t0) & (m < t1)]
    return np.nan if len(m) < min_marcas else 60.0 / np.median(np.diff(m))

def marcas_banda(rsp, fs=1000):
    x = decimate(decimate(rsp, 10, zero_phase=True), 4, zero_phase=True)
    f = fs / 40
    b, a = butter(2, [0.05, 1.0], btype="band", fs=f)
    x = filtfilt(b, a, x)
    iqr = np.subtract(*np.percentile(x, [75, 25]))
    pk, _ = find_peaks(x, distance=int(f), prominence=0.25 * iqr)
    return pk / f

def _lista(c):
    return np.array([]) if pd.isna(c) or str(c).strip() == "" else np.array(str(c).split(), dtype=float)

filas, t_ini = [], time.time()

def agregar(base, sujeto, est, refs):
    """refs: lista de (t0, RR de referencia, info extra). Solo se estiman ventanas con referencia fiable."""
    for t0, ref, extra in refs:
        filas.append({"Base": base, "Sujeto": f"{base}:{sujeto}", "Inicio (s)": t0, "RR referencia": ref,
                      **extra, **est.ventana(t0)})

# CASE (referencia: banda; fiable si el conteo y la FFT de la banda coinciden ≤ 2 rpm)
for p in sorted(CASE_DIR.glob("sub_*.csv"), key=lambda p: int(p.stem.split("_")[1])):
    d = pd.read_csv(p, usecols=["bvp", "rsp", "video"])
    vid = d["video"].to_numpy(int)
    est = rg.Estimador(d["bvp"].to_numpy(float), 1000, P, factores_hp=FACTORES)
    rsp = d["rsp"].to_numpy(float)
    marcas = marcas_banda(rsp)
    rsp5 = decimate(decimate(resample_poly(rsp, FS, 1000), 5, zero_phase=True), 5, zero_phase=True)
    refs = []
    for t0 in est.inicios():
        v = vid[int(t0 * 1000):int((t0 + VENTANA_S) * 1000)]
        if len(v) == 0 or v.min() != v.max() or v[0] not in ACTIVACION:
            continue
        r = rr_ref(marcas, t0, t0 + VENTANA_S)
        seg = rsp5[int(t0 * 5):int((t0 + VENTANA_S) * 5)]
        if not np.isfinite(r) or not P["rpm_min"] <= r <= P["rpm_max"] or len(seg) < VENTANA_S * 5:
            continue
        f, Pw = rg.espectro(seg, 5, 2048)
        bnd = (f >= P["rpm_min"]) & (f <= P["rpm_max"])
        if abs(r - f[bnd][np.argmax(Pw[bnd])]) <= 2:
            refs.append((t0, r, {"Activación": ACTIVACION[int(v[0])]}))
    agregar("CASE", p.stem, est, refs)
print(f"CASE listo ({time.time() - t_ini:.0f} s)")

# BIDMC (referencia: respiraciones anotadas; fiable si los dos anotadores coinciden ≤ 2 rpm)
for i in range(1, 54):
    ruta = BASE_DIR / "data" / f"bidmc{i:02d}"
    if not ruta.with_suffix(".hea").exists():
        continue
    rec = wfdb.rdrecord(str(ruta))
    ppg = rec.p_signal[:, [s.strip(" ,").upper() for s in rec.sig_name].index("PLETH")]
    ann = wfdb.rdann(str(ruta), "breath")
    aux = np.array([(x or "").strip(" \x00") for x in ann.aux_note])
    m1, m2 = (np.sort(ann.sample[aux == k]) / (ann.fs or rec.fs) for k in ("ann1", "ann2"))
    est = rg.Estimador(ppg, rec.fs, P, factores_hp=FACTORES)
    refs = []
    for t0 in est.inicios():
        r1, r2 = rr_ref(m1, t0, t0 + VENTANA_S), rr_ref(m2, t0, t0 + VENTANA_S)
        if np.isfinite(r1) and abs(r1 - r2) <= CFG["umbral_anot"]:
            refs.append((t0, r1, {}))
    agregar("BIDMC", ruta.name, est, refs)
print(f"BIDMC listo ({time.time() - t_ini:.0f} s)")

# CapnoBase (referencia: capnografía)
for fs_ in sorted(CAPNO_DIR.glob("*_signal.csv")):
    rid = fs_.name.replace("_signal.csv", "")
    par = pd.read_csv(CAPNO_DIR / f"{rid}_param.csv").iloc[0]
    lab = pd.read_csv(CAPNO_DIR / f"{rid}_labels.csv", keep_default_na=False).iloc[0]
    est = rg.Estimador(pd.read_csv(fs_)["pleth_y"].to_numpy(float), int(par["samplingrate_pleth"]), P,
                       factores_hp=FACTORES)
    m1 = np.sort(_lista(lab["co2_startexp_x"])) / int(par["samplingrate_co2"])
    refs = [(t0, r, {}) for t0 in est.inicios() if np.isfinite(r := rr_ref(m1, t0, t0 + VENTANA_S))]
    agregar("CapnoBase", rid, est, refs)
print(f"CapnoBase listo ({time.time() - t_ini:.0f} s)")

D = pd.DataFrame(filas)
D["Válida"] = ~D["Artefacto"]
for fac in FACTORES:
    for tol in TOLS:
        D[f"F {fac} {tol:g}"] = rg.fusion(D["S"], D[f"Da {fac}"], tol)
D.to_csv(OUT / "rr_generalizable_ventanas.csv", index=False)
print(D.groupby("Base")["Válida"].agg(["size", "sum"]).rename(columns={"size": "ventanas", "sum": "válidas"}))

In [ ]:
# g-3 — Métricas
def metricas(g, col):
    ok = g[col].notna()
    if ok.sum() == 0:
        return {"Cobertura %": 0.0}
    h = g[ok]
    err = h[col] - h["RR referencia"]
    e = err.abs()
    r = h.groupby("Sujeto").apply(lambda s: s[col].corr(s["RR referencia"])
                                  if s[col].std() > 0 and s["RR referencia"].std() > 0 else np.nan)
    out = {"MAE mediano": e.groupby(h["Sujeto"]).mean().median(), "MAE medio": e.mean(),
           "RMSE": np.sqrt((err ** 2).mean()), "Sesgo": err.mean(),
           "LdA inf": err.mean() - 1.96 * err.std(), "LdA sup": err.mean() + 1.96 * err.std(),
           "% ≤ 2 rpm": 100 * (e <= 2).mean(), "Cobertura %": 100 * ok.mean(),
           "r dentro de sujeto": r.median()}
    if "Activación" in h and h["Activación"].notna().any():
        dl = h.groupby(["Sujeto", "Activación"])[[col, "RR referencia"]].mean().unstack()
        if (col, 1) in dl and (col, 0) in dl:
            dm, dr = dl[(col, 1)] - dl[(col, 0)], dl[("RR referencia", 1)] - dl[("RR referencia", 0)]
            out.update({"Δ alta−baja método": dm.median(), "Δ alta−baja ref": dr.median(), "r de los Δ": dm.corr(dr)})
    return out

PRINCIPALES = {"A: FFT original": "A", "D: conteo fijo 0.17 Hz": "D", "S: pico sobre el fondo": "S",
               "Da: conteo adaptativo": "Da", "F: fusión S + Da": f"F {P['hp_factor']} {P['tol_fusion_rpm']:g}"}
V = D[D["Válida"]]
TAB = pd.concat({base: pd.DataFrame({n: metricas(V[V["Base"] == base], c) for n, c in PRINCIPALES.items()}).T
                 for base in ["CASE", "BIDMC", "CapnoBase"] if (V["Base"] == base).any()}, names=["Base", "Método"])
TAB.to_csv(OUT / "rr_generalizable_resumen.csv")
print(TAB.round(2).to_string())

In [ ]:
# g-4 — Sensibilidad a los parámetros (MAE mediano / cobertura por base)
filas = []
for fac in FACTORES:
    for tol in TOLS:
        for base, g in V.groupby("Base"):
            m = metricas(g, f"F {fac} {tol:g}")
            filas.append({"factor pasa-alto": fac, "tolerancia": tol, "Base": base,
                          "MAE mediano": m.get("MAE mediano"), "Cobertura %": m["Cobertura %"]})
SENS = pd.DataFrame(filas).pivot_table(index=["factor pasa-alto", "tolerancia"], columns="Base",
                                       values=["MAE mediano", "Cobertura %"])
print("Fusión F con distintos parámetros:")
print(SENS.round(2).to_string())
print("\nConteo adaptativo Da (MAE mediano por base):")
print(pd.DataFrame({fac: {b: metricas(g, f"Da {fac}").get("MAE mediano") for b, g in V.groupby("Base")}
                    for fac in FACTORES}).round(2).to_string())

In [ ]:
# g-5 — Validación dejando una base fuera
# Regla: con las dos bases de entrenamiento se elige (factor, tolerancia) con menor MAE mediano promedio
# entre las combinaciones con cobertura ≥ 70 % en ambas; se evalúa en la base excluida.
bases = sorted(V["Base"].unique())
filas = []
for fuera in bases:
    entr = [b for b in bases if b != fuera]
    mejor, puntaje = None, np.inf
    for fac in FACTORES:
        for tol in TOLS:
            ms = [metricas(V[V["Base"] == b], f"F {fac} {tol:g}") for b in entr]
            if all(m["Cobertura %"] >= 70 for m in ms) and np.mean([m["MAE mediano"] for m in ms]) < puntaje:
                mejor, puntaje = (fac, tol), np.mean([m["MAE mediano"] for m in ms])
    if mejor is None:
        mejor = (P["hp_factor"], P["tol_fusion_rpm"])
    m_sel = metricas(V[V["Base"] == fuera], f"F {mejor[0]} {mejor[1]:g}")
    m_pri = metricas(V[V["Base"] == fuera], PRINCIPALES["F: fusión S + Da"])
    filas.append({"Base excluida": fuera, "Elegido con las otras dos": f"{mejor[0]} / ±{mejor[1]:g}",
                  "MAE mediano (elegido)": m_sel.get("MAE mediano"), "Cobertura % (elegido)": m_sel["Cobertura %"],
                  "MAE mediano (fijado a priori)": m_pri.get("MAE mediano"), "Cobertura % (a priori)": m_pri["Cobertura %"]})
LODO = pd.DataFrame(filas)
LODO.to_csv(OUT / "rr_generalizable_lodo.csv", index=False)
print(LODO.round(2).to_string(index=False))
(OUT / "config_generalizable.json").write_text(json.dumps({**P, "fondo_rpm": list(P["fondo_rpm"])}, indent=2))

In [ ]:
# g-6 — Figura: Bland-Altman, FFT original frente a fusión
bases_fig = [b for b in ["CASE", "BIDMC", "CapnoBase"] if (V["Base"] == b).any()]
fig, axs = plt.subplots(2, len(bases_fig), figsize=(4.2 * len(bases_fig), 7), sharey=True, squeeze=False)
for j, base in enumerate(bases_fig):
    g = V[V["Base"] == base]
    for i, (nombre, col) in enumerate([("A: FFT original", "A"), ("F: fusión S + Da", PRINCIPALES["F: fusión S + Da"])]):
        h = g[g[col].notna()]
        media, dif = (h[col] + h["RR referencia"]) / 2, h[col] - h["RR referencia"]
        ax = axs[i, j]
        ax.scatter(media, dif, s=3, alpha=0.3)
        for y_, ls in [(dif.mean(), "-"), (dif.mean() + 1.96 * dif.std(), "--"), (dif.mean() - 1.96 * dif.std(), "--")]:
            ax.axhline(y_, c="k", ls=ls, lw=0.8)
        ax.set_title(f"{base} — {nombre}\nsesgo {dif.mean():+.2f}, LdA ±{1.96 * dif.std():.2f} rpm", fontsize=9)
        ax.set_xlabel("Media (rpm)")
        if j == 0:
            ax.set_ylabel("Método − referencia (rpm)")
plt.tight_layout()
plt.savefig(OUT / "rr_generalizable_bland_altman.png", dpi=300)
plt.show()

In [ ]:
# g-7 — Figura: una ventana de CASE donde la FFT original cae en el borde y S encuentra la respiración
c = V[(V["Base"] == "CASE") & (V["A"] < 7) & ((V["S"] - V["RR referencia"]).abs() <= 1)]
if len(c):
    w = c.iloc[len(c) // 2]
    sujeto = w["Sujeto"].split(":")[1]
    d = pd.read_csv(CASE_DIR / f"{sujeto}.csv", usecols=["bvp"])
    est = rg.Estimador(d["bvp"].to_numpy(float), 1000, P)
    mod = min(["RIIV", "RIAV", "RIFV"], key=lambda k: abs(w[f"S {k}"] - w["RR referencia"]) if np.isfinite(w[f"S {k}"]) else 99)
    serie = rg.serie_4hz(est.t, est.mods[mod], w["Inicio (s)"])
    rr_s, _, det = rg.pico_sobre_fondo(serie, P, devolver_detalle=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogy(det["f"], det["P"], lw=1, label=f"Espectro de {mod}")
    ax.semilogy(det["f_fondo"], det["fondo"], "k--", lw=1, label="Fondo ajustado (1/f)")
    ax.axvline(w["RR referencia"], c="g", lw=2, alpha=0.5, label=f"Banda: {w['RR referencia']:.1f} rpm")
    ax.axvline(w["A"], c="r", ls=":", label=f"FFT original (A): {w['A']:.1f} rpm")
    ax.axvline(rr_s, c="b", ls="--", label=f"Pico sobre el fondo (S): {rr_s:.1f} rpm")
    ax.set_xlim(2, 40)
    vis = (det["f"] >= 2) & (det["f"] <= 40)
    ax.set_ylim(det["P"][vis].min() / 3, det["P"][vis].max() * 3)
    ax.set_xlabel("Frecuencia (rpm)")
    ax.set_ylabel("Potencia")
    ax.set_title(f"CASE {sujeto}, t = {w['Inicio (s)']:.0f} s", fontsize=9)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT / "rr_generalizable_ejemplo_case.png", dpi=300)
    plt.show()
else:
    print("No hay una ventana de ejemplo con ese criterio.")